# Título: Análisis descriptivo con segmentación temporal

### Installs

In [ ]:
# %pip install seaborn

### Imports

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data_cleaning_2026_20_04.csv")

In [ ]:
df.info()

Transformación y reducción de datos

In [ ]:
df_clean = df.loc[:, ~df.columns.isin(['Pet', 'Weight', 'Height'])]


In [ ]:
df_clean['Month_absence'].unique()

In [ ]:
df_clean['Month_absence'] = df_clean['Month_absence'].astype('Int64')
df_clean.info()

In [ ]:
df_clean[df_clean['Absenteeism_hours'] == 0]

In [ ]:
df_clean[df_clean['Month_absence'].isna()]

Decidimos quitar los valores faltante por el tipo de datos, al no representar un ausencia real por el tipo de dataset.    

In [ ]:
df_clean = df_clean[df_clean['Absenteeism_hours'] > 0]
df_clean.info()

In [ ]:
df_clean.to_csv("data_transformation_2026_22_04.csv", index=False, encoding="utf-8")

### Magnitud  del ausentísmo

In [ ]:
df_clean['Absenteeism_hours'].describe()

In [ ]:
tot_hours_ausent = df_clean['Absenteeism_hours'].sum()
print(f"Total hours of absenteeism: {tot_hours_ausent}")

""" asumiendo que son 36 empleados con 40 horas semanales, 52 semanas y 22 dias feriados. Calculamos el total de horas laborales al año y luego el porcentaje de horas ausentes """

total_employees = 36
hours_per_week = 40
weeks_per_year = 52
holidays_per_year = 22
total_working_hours_per_year = (hours_per_week * weeks_per_year - holidays_per_year * hours_per_week / 5) * total_employees
percentage_absent = (tot_hours_ausent / total_working_hours_per_year) * 100
print(f"Percentage of hours absent: {percentage_absent:.2f}%")

In [ ]:
def classify_hours(h):
    if pd.isna(h):
        return np.nan
    if h <= 4:
        return 'Very Short (0-4h)'
    if h <= 8:
        return 'Short (<=8h)'
    elif h <= 24:
        return 'Medium (9-24h)'
    else:
        return 'Long (>24h)'

df_clean['Absenteeism_terms'] = df_clean['Absenteeism_hours'].apply(classify_hours)

print(df_clean['Absenteeism_terms'].value_counts(dropna=False))

In [ ]:

counts = df_clean['Absenteeism_terms'].fillna('Missing').value_counts(dropna=False)
labels_with_counts = [f"{lab} ({cnt})" for lab, cnt in counts.items()]

# Palette migliorata (verde, giallo, arancio, grigio)
colors = ['#719b77', "#b0b997", '#c3b7a3', '#c9916c'][:len(counts)]

plt.figure(figsize=(6,6))
plt.pie(counts, labels=labels_with_counts, autopct='%1.1f%%', startangle=90, colors=colors)
plt.axis('equal')
plt.show()


In [ ]:
# Histograma de distribución general

plt.figure(figsize=(8,5))
sns.histplot(df_clean['Absenteeism_hours'], bins=20, kde=True)
plt.title('Distribución general de horas de absentismo')
plt.xlabel('Horas de absentismo')
plt.ylabel('Frecuencia')
plt.show()

Media inflada por casos extremos (120 h) - mean 7.6h y median 4h
Gran dispersion por presencia de casos extremos

Magnitud moderada en  promedio 7.6h pero altamente desigual: mediana de 4h y 75% con menos de 8h
Outliers incrementan la media y explica alta variabilidad.
Los percentiles son más adecuados para describir el comportamiento típico del absentísmo y no la media 

In [ ]:
# Sumar horas totales por categoria
hours_by_term = df_clean.groupby('Absenteeism_terms')['Absenteeism_hours'].sum()

# Total general
total_hours = df_clean['Absenteeism_hours'].sum()

# Porcentages
percent_long = (hours_by_term.get('Long (>24h)', 0) / total_hours) * 100
percent_medium = (hours_by_term.get('Medium (8-24h)', 0) / total_hours) * 100
percent_short = (hours_by_term.get('Short (<=8h)', 0) / total_hours) * 100
percent_very_short = (hours_by_term.get('Very Short (0-4h)', 0) / total_hours) * 100

print("Horas totales por categoria:")
print(hours_by_term)
print("\nPercentage casos largos:", round(percent_long, 2), "%")
print("\nPorcentage casos médios:", round(percent_medium, 2), "%")
print("\nPorcentage casos cortos:", round(percent_short, 2), "%")
print("\nPorcentage casos muy cortos:", round(percent_very_short, 2), "%")

In [ ]:
hours_by_term = hours_by_term.sort_values(ascending=False)

labels = [f"{lab} ({val})" for lab, val in hours_by_term.items()]
values = hours_by_term.values

colors = ['#c9916c', '#c3b7a3', '#b0b997', '#719b77'][:len(values)]

plt.figure(figsize=(7,7))
plt.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors
)
plt.title('Distribución de horas de absentismo por tipo de ausencia')
plt.axis('equal')
plt.show()

### Visualizar patrones temporales

Identificar meses criticos, estacionalidad y variabilidad

In [ ]:
# Agrupación por mes

df_month = df_clean.groupby('Month_absence')['Absenteeism_hours'].agg(
    total_absence='sum',
    mean_absence='mean',
    median_absence='median',
    count='count'
).reset_index()

df_month


In [ ]:
# Gráfico de barras: total de horas por mes (meses criticos)

plt.figure(figsize=(10,5))
sns.barplot(data=df_month,x='Month_absence', y='total_absence')
plt.title('Total de horas de absentismo por mes')
plt.xlabel('Mes')
plt.ylabel('Horas totales')
plt.show()

In [ ]:
# boxplot por mes para ver dispersion y outliers (variabilidad)

plt.figure(figsize=(12,5))
sns.boxplot(data=df_clean, x='Month_absence', y='Absenteeism_hours')
plt.title('Distribución del absentismo por mes')
plt.xlabel('Mes')
plt.ylabel('Horas de absentismo')
plt.show()

### Variación estacional

In [ ]:
# Grafico por estacion - (estacionalidad)

# Agrupación por estación
df_season = df_clean.groupby('Seasons')['Absenteeism_hours'].agg(
    total_absence='sum',
    mean_absence='mean',
    median_absence='median',
    count='count'
).reset_index()

df_season

In [ ]:
# Gráfico comparativo por estación

plt.figure(figsize=(8,5))
sns.barplot(data=df_season, x='Seasons', y='total_absence')
plt.title('Total de horas de absentismo por estación')
plt.xlabel('Estación')
plt.ylabel('Horas totales')
plt.show()

Más riesgo de ausencias prolongadas en invierno, verano y primavera con niveles similares 

### Perfiles de riesgo asociados al tiempo

In [ ]:
# Cruzamos estación y tipo de ausencia
 
df_cross = df_clean.groupby(['Seasons', 'Reason_absence_mapped'])['Absenteeism_hours'].sum().reset_index()

# motivos por estación

df_cross.groupby('Seasons').apply(
    lambda x: x.sort_values('Absenteeism_hours', ascending=False).head(3)
)

### Identificación meses críticos

In [ ]:
# Top 3 meses con mayor absentismo total

df_month.sort_values('total_absence', ascending=False).head(3)

In [ ]:
# Top 3 meses con mayor absentismo promedio

df_month.sort_values('mean_absence', ascending=False).head(3)